# 🚀 XAUUSD M1 Machine Learning Pipeline สำหรับใช้งานร่วมกับ EA
### วัตถุประสงค์:
- ทำนายว่าในอีก **N แท่งข้างหน้า** ถ้าราคาขึ้น จะมีโอกาสวิ่งขึ้นได้สูงสุดเท่าไหร่ (Max Upside)
- และถ้าราคาลง จะมีโอกาสวิ่งลงได้ลึกสุดเท่าไหร่ (Max Downside)
- **แก้ปัญหาความต่างของระดับราคา (4000 vs 2000) 100%:** ทุก Feature และ Target ถูกแปลงเป็น **Stationary Scale-Invariant** (วัดเป็นเท่าของ ATR, Log Returns, และ Oscillators) ทำให้โมเดลเรียนรู้โครงสร้างความผันผวนและการไปต่อได้แม่นยำ ไม่ว่าทองคำจะอยู่ที่ราคาเท่าไหร่

In [ ]:
# 1. ติดตั้งไลบรารีที่จำเป็นบน Google Colab
!pip install -q lightgbm scikit-learn matplotlib seaborn joblib

In [ ]:
# 2. เชื่อมต่อ Google Drive (หรือระบุที่อยู่ไฟล์ใน Colab)
from google.colab import drive
import os

drive.mount('/content/drive')

# ระบุพาธไฟล์ CSV ที่เก็บไว้บน Google Drive
# ตัวอย่าง: '/content/drive/MyDrive/EA/XAUUSD.iux_M1_20200102_to_20251230.csv'
CSV_PATH = '/content/drive/MyDrive/EA/XAUUSD.iux_M1_20200102_to_20251230.csv'

# หากอัปโหลดไฟล์ตรงๆ เข้า Colab (ไม่ต้องผ่าน Drive):
if not os.path.exists(CSV_PATH):
    CSV_PATH = 'XAUUSD.iux_M1_20200102_to_20251230.csv'
    print('ใช้ไฟล์ใน Current Directory:', CSV_PATH)

In [ ]:
# 3. ตั้งค่าพารามิเตอร์การเทรน
N_FORWARD = 15          # จำนวนแท่งข้างหน้าที่ต้องการให้โมเดลทำนาย (เช่น 15 แท่ง)
RESAMPLE_TF = None      # ตั้งเป็น None เพื่อใช้ M1 หรือใส่ '5T' สำหรับ M5, '15T' สำหรับ M15 (ช่วยลด noise และเร็วขึ้น)
TRAIN_RATIO = 0.80      # แบ่งข้อมูล 80% แรกสำหรับ Train (2020-2024), 20% หลังสำหรับ Test (2025)

print(f"Target Horizon: อีก {N_FORWARD} แท่งข้างหน้า")
print(f"Timeframe: {'M1 ดั้งเดิม' if RESAMPLE_TF is None else RESAMPLE_TF}")

In [ ]:
# 4. โหลดและจัดการหน่วยความจำ (Memory Optimization)
import pandas as pd
import numpy as np
import gc

dtypes = {
    'open': 'float32',
    'high': 'float32',
    'low': 'float32',
    'close': 'float32',
    'tick_volume': 'int32',
    'spread': 'int16'
}

print("กำลังโหลด Dataset...")
df = pd.read_csv(CSV_PATH, usecols=['datetime', 'open', 'high', 'low', 'close', 'tick_volume', 'spread'], dtype=dtypes)
df['datetime'] = pd.to_datetime(df['datetime'])
df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)

if RESAMPLE_TF is not None:
    print(f"กำลัง Resample ข้อมูลเป็น {RESAMPLE_TF}...")
    df.set_index('datetime', inplace=True)
    df = df.resample(RESAMPLE_TF).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'tick_volume': 'sum',
        'spread': 'mean'
    }).dropna().reset_index()

print(f"ข้อมูลพร้อมใช้งานทั้งหมด: {len(df):,} แถว")
df.head()

In [ ]:
# 5. ฟังก์ชันสร้าง Feature & Target แบบ Scale-Invariant
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0.0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100.0 - (100.0 / (1.0 + rs))

def extract_features_and_targets(df, n_forward=15):
    # คำนวณ True Range และ ATR
    high_low = df['high'] - df['low']
    high_close = (df['high'] - df['close'].shift(1)).abs()
    low_close = (df['low'] - df['close'].shift(1)).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr14 = tr.rolling(14).mean()
    atr50 = tr.rolling(50).mean()
    safe_atr = atr14.replace(0, np.nan)
    
    X = pd.DataFrame(index=df.index)
    
    # 1. Normalized Momentum & Returns (Stationary)
    for lag in [1, 3, 5, 15, 30, 60]:
        X[f'ret_{lag}'] = np.log(df['close'] / df['close'].shift(lag))
        
    # 2. Moving Average Distances Normalized by ATR (ไร้หน่วยราคาดิบ)
    ema20 = df['close'].ewm(span=20, adjust=False).mean()
    ema50 = df['close'].ewm(span=50, adjust=False).mean()
    ema200 = df['close'].ewm(span=200, adjust=False).mean()
    X['dist_ema20_atr'] = (df['close'] - ema20) / safe_atr
    X['dist_ema50_atr'] = (df['close'] - ema50) / safe_atr
    X['dist_ema200_atr'] = (df['close'] - ema200) / safe_atr
    
    # 3. Volatility Ratios
    X['norm_atr14'] = atr14 / df['close']
    X['atr_ratio'] = atr14 / (atr50 + 1e-9)
    
    # 4. Candlestick Structure per ATR
    X['bar_range_atr'] = (df['high'] - df['low']) / safe_atr
    X['bar_body_atr'] = (df['close'] - df['open']) / safe_atr
    X['upper_wick_atr'] = (df['high'] - np.maximum(df['open'], df['close'])) / safe_atr
    X['lower_wick_atr'] = (np.minimum(df['open'], df['close']) - df['low']) / safe_atr
    
    # 5. Oscillators & Volume
    X['rsi14'] = calculate_rsi(df['close'], period=14) / 100.0
    safe_vol = df['tick_volume'].replace(0, 1)
    X['vol_ratio_20'] = safe_vol / (safe_vol.rolling(20).mean() + 1e-9)
    X['vol_ratio_100'] = safe_vol / (safe_vol.rolling(100).mean() + 1e-9)
    
    # 6. Session Cyclic Encoding
    hour = df['datetime'].dt.hour + df['datetime'].dt.minute / 60.0
    dow = df['datetime'].dt.dayofweek
    X['hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
    X['hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
    X['dow_sin'] = np.sin(2 * np.pi * dow / 5.0)
    X['dow_cos'] = np.cos(2 * np.pi * dow / 5.0)
    
    # 7. Target Calculation (Max Upside / Downside in ATR units)
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=n_forward)
    future_max_high = df['high'].shift(-1).rolling(window=indexer).max()
    future_min_low = df['low'].shift(-1).rolling(window=indexer).min()
    
    y = pd.DataFrame(index=df.index)
    y['target_upside_atr'] = np.maximum(0.0, future_max_high - df['close']) / safe_atr
    y['target_downside_atr'] = np.maximum(0.0, df['close'] - future_min_low) / safe_atr
    
    # Clip extreme flash crash outliers
    y['target_upside_atr'] = y['target_upside_atr'].clip(upper=25.0)
    y['target_downside_atr'] = y['target_downside_atr'].clip(upper=25.0)
    
    return X, y, atr14

print("กำลังคำนวณ Features และ Targets...")
X, y, atr14 = extract_features_and_targets(df, n_forward=N_FORWARD)

# ตัดแถว NaN ที่เกิดจาก Rolling Window
valid_mask = ~(X.isna().any(axis=1) | y.isna().any(axis=1))
X_clean = X[valid_mask].astype('float32')
y_clean = y[valid_mask].astype('float32')
df_clean = df[valid_mask]

print(f"จำนวนข้อมูลพร้อมเทรนหลังกรอง NaN: {len(X_clean):,} แถว")
print("คอลัมน์ Features ทั้งหมด:", list(X_clean.columns))

In [ ]:
# 6. แบ่งข้อมูล Train / Validation ตามลำดับเวลา (Time-Series Split)
split_idx = int(len(X_clean) * TRAIN_RATIO)

X_train, X_test = X_clean.iloc[:split_idx], X_clean.iloc[split_idx:]
y_train, y_test = y_clean.iloc[:split_idx], y_clean.iloc[split_idx:]

print(f"Train set: {len(X_train):,} แถว (ตั้งแต่ {df_clean.iloc[0]['datetime']} ถึง {df_clean.iloc[split_idx]['datetime']})")
print(f"Test set:  {len(X_test):,} แถว (ตั้งแต่ {df_clean.iloc[split_idx]['datetime']} ถึง {df_clean.iloc[-1]['datetime']})")

In [ ]:
# 7. เทรนโมเดล LightGBM สำหรับทำนาย Upside (โอกาสที่ราคาจะขึ้นไปได้สูงสุดกี่เท่าของ ATR)
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score

lgb_params = {
    'objective': 'regression_l1',  # MAE loss (ทนต่อ Outliers ได้ดีกว่า MSE)
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1
}

print("--- กำลังเทรน Model 1: Upside Predictor ---")
model_upside = lgb.LGBMRegressor(**lgb_params)
model_upside.fit(
    X_train, y_train['target_upside_atr'],
    eval_set=[(X_test, y_test['target_upside_atr'])],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=50)]
)

print("\n--- กำลังเทรน Model 2: Downside Predictor ---")
model_downside = lgb.LGBMRegressor(**lgb_params)
model_downside.fit(
    X_train, y_train['target_downside_atr'],
    eval_set=[(X_test, y_test['target_downside_atr'])],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=50)]
)

In [ ]:
# 8. ประเมินผลโมเดลบน Test Set
pred_upside = model_upside.predict(X_test)
pred_downside = model_downside.predict(X_test)

print("=== ประเมินผลโมเดลบนข้อมูลทดสอบ (Out-of-Time Test Set) ===")
print(f"Upside MAE:   {mean_absolute_error(y_test['target_upside_atr'], pred_upside):.4f} ATR")
print(f"Downside MAE: {mean_absolute_error(y_test['target_downside_atr'], pred_downside):.4f} ATR")

# Correlation ระหว่างค่าจริงกับค่าที่โมเดลทำนาย
corr_up = np.corrcoef(y_test['target_upside_atr'], pred_upside)[0, 1]
corr_down = np.corrcoef(y_test['target_downside_atr'], pred_downside)[0, 1]
print(f"Upside Correlation:   {corr_up * 100:.2f}%")
print(f"Downside Correlation: {corr_down * 100:.2f}%")

In [ ]:
# 9. แสดง Feature Importance (ตัวแปรไหนส่งผลต่อการไปต่อมากที่สุด)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
lgb.plot_importance(model_upside, max_num_features=10, ax=axes[0], title='Top 10 Features (Upside Model)')
lgb.plot_importance(model_downside, max_num_features=10, ax=axes[1], title='Top 10 Features (Downside Model)')
plt.tight_layout()
plt.show()

In [ ]:
# 10. ตัวอย่าง: การนำผลทำนายไปใช้จริงใน EA
"""
ใน EA ไม่ว่าจะราคา 2000 หรือ 4000:
1. โมเดลทำนายค่าออกมาเป็น 'จำนวนเท่าของ ATR' เช่น pred_up = 2.4 ATR
2. EA อ่านค่า ATR14 ปัจจุบันจากกราฟ (เช่น ATR = $3.50)
3. ระยะ Upside ที่คาดหวังเป็นดอลลาร์ = 2.4 * 3.50 = $8.40
4. ราคาสูงสุดที่คาดหวัง = Current_Close + $8.40
"""
sample_idx = -100
sample_row = df_clean.iloc[split_idx + sample_idx]
curr_price = sample_row['close']
curr_atr = atr14.iloc[split_idx + sample_idx]
sample_features = X_test.iloc[[sample_idx]]

est_up_atr = model_upside.predict(sample_features)[0]
est_down_atr = model_downside.predict(sample_features)[0]

est_up_price = curr_price + (est_up_atr * curr_atr)
est_down_price = curr_price - (est_down_atr * curr_atr)

print(f"เวลาทดสอบ: {sample_row['datetime']}")
print(f"ราคาปิดปัจจุบัน: {curr_price:,.2f} | ATR14: {curr_atr:.2f}")
print(f"-> คาดการณ์โอกาสขึ้นได้: +{est_up_atr:.2f} ATR (ประมาณ +${est_up_atr * curr_atr:.2f}) -> เป้าหมาย: {est_up_price:,.2f}")
print(f"-> คาดการณ์โอกาสลงได้: -{est_down_atr:.2f} ATR (ประมาณ -${est_down_atr * curr_atr:.2f}) -> เป้าหมาย: {est_down_price:,.2f}")

In [ ]:
# 11. บันทึก Model สำหรับนำไปใช้กับ EA หรือเชื่อมต่อผ่าน API
import joblib

joblib.dump(model_upside, 'xauusd_upside_model.joblib')
joblib.dump(model_downside, 'xauusd_downside_model.joblib')

# หรือบันทึกเป็น Native LightGBM Model (.txt)
model_upside.booster_.save_model('xauusd_upside_model.txt')
model_downside.booster_.save_model('xauusd_downside_model.txt')

print("✅ บันทึกโมเดลเรียบร้อยแล้ว: xauusd_upside_model.joblib และ xauusd_downside_model.joblib")